# Silver Layer — Transaction Data

**Runs in two modes automatically:**

| Mode | Bronze reads from | Silver writes to |
|------|------------------|-----------------|
| **Google Colab** | `MyDrive/FinPulse/data/bronze/transactions` | `MyDrive/FinPulse/data/silver/transactions` |
| **Local (VS Code)** | `<project_root>/data/bronze/transactions` | `<project_root>/data/silver/transactions` |

Override any path via environment variables: `BRONZE_TRANSACTIONS_PATH`, `SILVER_OUTPUT_PATH`, `ICEBERG_WAREHOUSE`, `GDRIVE_FINPULSE_ROOT`.

In [1]:
import sys
print("Python version:", sys.version)
print("Setup verified ✅")

Python version: 3.11.1 (tags/v3.11.1:a7a450f, Dec  6 2022, 19:58:39) [MSC v.1934 64 bit (AMD64)]
Setup verified ✅


In [2]:
# Dependencies should be managed via requirements.txt or Docker environment
# e.g. pip install pyspark pyiceberg pandas python-dotenv

In [3]:
# ── Verify Bronze data exists locally ────────────────────────────────────────────
import os, sys as _sys
from pathlib import Path

def _find_project_root(marker: str = "pyproject.toml") -> Path:
    current = Path(os.path.abspath("")).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    return current.parent  # fallback: one level above notebooks/

PROJECT_ROOT = _find_project_root()
BRONZE_BASE = str(PROJECT_ROOT / "data" / "bronze" / "transactions")

# Skip check on Colab — bronze is on Google Drive, path is configured in the setup cell
_on_colab = "google.colab" in _sys.modules or os.path.exists("/content")

if _on_colab:
    print("ℹ️  Colab detected — skipping local Bronze check (path configured in setup cell)")
elif not os.path.exists(BRONZE_BASE):
    print(f"⚠️  Bronze folder not found locally: {BRONZE_BASE}")
    print("   Set BRONZE_TRANSACTIONS_PATH env var or ensure data/bronze/transactions exists.")
else:
    partitions = [d for d in os.listdir(BRONZE_BASE) if d.startswith("date=")]
    if not partitions:
        print(f"⚠️  No date= partitions found in {BRONZE_BASE}")
    else:
        latest = sorted(partitions)[-1]
        BRONZE_FILE = os.path.join(BRONZE_BASE, latest, "transactions_raw.csv")
        if os.path.exists(BRONZE_FILE):
            size_mb = os.path.getsize(BRONZE_FILE) / (1024 * 1024)
            print(f"✅ Bronze file found: {BRONZE_FILE}")
            print(f"   Size: {size_mb:.1f} MB")
            print(f"   Partition: {latest}")
        else:
            print(f"⚠️  Expected file not found: {BRONZE_FILE}")

print(f"ℹ️  Local bronze base: {BRONZE_BASE} (BRONZE_TRANSACTIONS_PATH set in setup cell)")


✅ Bronze file found: C:\FinPulse Project\data\bronze\transactions\date=2026-03-29\transactions_raw.csv
   Size: 712.4 MB
   Partition: date=2026-03-29
ℹ️  Local bronze base: C:\FinPulse Project\data\bronze\transactions (BRONZE_TRANSACTIONS_PATH set in setup cell)


In [4]:
# ============================================================
# FinPulse - Silver Layer: Transaction Data (8GB RAM SAFETY v5.2)
# Purpose: Total Stability Optimization for 6.3M records
# ============================================================
import os
import sys
from pathlib import Path
from datetime import datetime

# -- STEP 1: Environment Logic ---------------------------
def _find_project_root(marker='pyproject.toml') -> Path:
    curr = Path(os.getcwd()).resolve()
    for cand in [curr] + list(curr.parents):
        if (cand / marker).exists(): return cand
    return curr

PROJECT_ROOT = _find_project_root()

# STABILITY FIX 5.2: Move Spark Temp OUTSIDE project folder to prevent 
# Google Drive sync app from locking shuffle files during renames.
TEMP_DIR = Path.home() / 'FinPulse_Spark_Temp'
os.makedirs(TEMP_DIR, exist_ok=True)

# JIT Stability Fix: Create sidecar file to avoid shell '<' issues
EXCLUDE_FILE = PROJECT_ROOT / 'exclude.txt'
with open(EXCLUDE_FILE, 'w') as f:
    f.write('exclude org/apache/iceberg/shaded/org/apache/arrow/memory/ArrowBuf <init>\n')

# Force clear invalid SPARK_HOME
if 'SPARK_HOME' in os.environ and not Path(os.environ['SPARK_HOME']).is_dir():
    del os.environ['SPARK_HOME']

# STABILITY FIX: Use Microsoft JDK 11 (Stable for Spark 3.5 on Windows)
JDK11_PATH = r'C:\Program Files\Microsoft\jdk-11.0.30.7-hotspot'
os.environ['JAVA_HOME'] = JDK11_PATH
os.environ['HADOOP_HOME'] = str(PROJECT_ROOT / 'hadoop')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Add bins to PATH to ensure hadoop.dll and java.exe are reachable
for p in [Path(JDK11_PATH)/'bin', Path(os.environ['HADOOP_HOME'])/'bin']:
    if p.is_dir() and str(p) not in os.environ['PATH']:
        os.environ['PATH'] = str(p) + os.pathsep + os.environ['PATH']

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, StringType, DoubleType, TimestampType
from dotenv import load_dotenv
load_dotenv(override=True)

# Path Resolution Logic (Local vs Google Drive)
_GDRIVE_ROOT = Path(os.environ.get('GDRIVE_FINPULSE_ROOT', 'G:/My Drive/FinPulse'))
if _GDRIVE_ROOT.exists():
    _DEF_BRONZE = str(_GDRIVE_ROOT / 'data' / 'bronze' / 'transactions')
    _DEF_SILVER = str(_GDRIVE_ROOT / 'data' / 'silver' / 'transactions')
else:
    _DEF_BRONZE = str(PROJECT_ROOT / 'data' / 'bronze' / 'transactions')
    _DEF_SILVER = str(PROJECT_ROOT / 'data' / 'silver' / 'transactions')

PIPELINE_VERSION = 'v1.0'
INGESTION_TIMESTAMP = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
BRONZE_PATH = os.environ.get('BRONZE_TRANSACTIONS_PATH', _DEF_BRONZE)
SILVER_OUTPUT_PATH = os.environ.get('SILVER_OUTPUT_PATH', _DEF_SILVER)

# IMPORTANT: ICEBERG WAREHOUSE MUST BE LOCAL ON C: TO AVOID VFS CRASHES
ICEBERG_WAREHOUSE = os.environ.get('ICEBERG_WAREHOUSE', str(PROJECT_ROOT / 'data' / 'silver' / 'iceberg_warehouse'))

print('=' * 40)
print(f'TOTAL STABILITY  : v5.2 (External Temp)')
print(f'JAVA_HOME         : {os.environ["JAVA_HOME"]}')
print(f'TEMP DIR (SAFE)   : {TEMP_DIR}')
print(f'WAREHOUSE (LOCAL) : {ICEBERG_WAREHOUSE}')
print('=' * 40)

# -- STEP 2: Spark Init ---------------------------
# Quote all paths manually to ensure Windows Space handling works 100%
spark_temp_path = str(TEMP_DIR).replace('\\', '/')
exclude_cfg_path = str(EXCLUDE_FILE).replace('\\', '/')
JVM_OPTS = (
    '-Dorg.apache.hadoop.io.nativeio.NativeIO.Windows.should_use_native_io=false'
    f' -Djava.io.tmpdir="{spark_temp_path}"'
    f' -XX:CompileCommandFile="{exclude_cfg_path}"'
    ' -XX:+UseG1GC'
)

spark = (
    SparkSession.builder
    .appName('FinPulse-Silver-Transactions')
    .master('local[2]')
    .config('spark.driver.memory', '2500m')
    .config('spark.executor.memory', '2500m')
    .config('spark.driver.extraJavaOptions', JVM_OPTS)
    .config('spark.executor.extraJavaOptions', JVM_OPTS)
    # Total Stability Overrides
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.sql.parquet.enableVectorizedReader', 'false')
    .config('spark.sql.iceberg.vectorization.enabled', 'false')
    .config('spark.hadoop.fs.file.impl', 'org.apache.hadoop.fs.LocalFileSystem')
    .config('spark.hadoop.fs.verify.checksum', 'false')
    .config('spark.network.timeout', '1200s')
    .config('spark.executor.heartbeatInterval', '150s')
    .config('spark.jars', str(PROJECT_ROOT / 'jars' / 'iceberg-spark-runtime-3.5_2.12-1.4.3.jar'))
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.local', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.local.type', 'hadoop')
    .config('spark.sql.catalog.local.warehouse', ICEBERG_WAREHOUSE)
    .config('spark.sql.defaultCatalog', 'local')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
print(f'Spark session ready | version {spark.version}')


TOTAL STABILITY  : v5.2 (External Temp)
JAVA_HOME         : C:\Program Files\Microsoft\jdk-11.0.30.7-hotspot
TEMP DIR (SAFE)   : C:\Users\amana\FinPulse_Spark_Temp
WAREHOUSE (LOCAL) : C:\FinPulse Project\data\silver\iceberg_warehouse
Spark session ready | version 3.5.8


In [5]:
def load_bronze_transactions(bronze_path: str):
    """
    Loads Bronze transaction data — handles all three storage layouts:

      Layout 1 (partitioned folder):  bronze_path/date=YYYY-MM-DD/transactions_raw.csv
      Layout 2 (flat folder):         bronze_path/transactions_raw.csv  (or any single .csv)
      Layout 3 (direct file path):    bronze_path ends with .csv
    """
    print("=" * 60)
    print("LOADING BRONZE TRANSACTION DATA")
    print("=" * 60)
    print(f"Source: {bronze_path}")

    p = Path(bronze_path)

    # ── Layout 3: path IS a CSV file ──────────────────────────
    if str(bronze_path).endswith(".csv"):
        if not p.exists():
            raise FileNotFoundError(f"CSV not found: {bronze_path}")
        file_path = str(p)
        print(f"Layout: direct CSV file")

    # ── Layout 1: partitioned by date= folders ─────────────────
    elif p.is_dir() and any(d.startswith("date=") for d in os.listdir(bronze_path)):
        partitions = sorted(d for d in os.listdir(bronze_path) if d.startswith("date="))
        latest = partitions[-1]
        file_path = str(p / latest / "transactions_raw.csv")
        print(f"Layout: partitioned  |  loading latest partition: {latest}")

    # ── Layout 2: flat folder with CSV inside ──────────────────
    else:
        csvs = sorted(p.glob("*.csv"))
        if not csvs:
            raise FileNotFoundError(
                f"No CSV files found in: {bronze_path}\n"
                f"Set BRONZE_TRANSACTIONS_PATH to the correct folder or file."
            )
        file_path = str(csvs[-1])  # pick alphabetically latest
        print(f"Layout: flat folder  |  loading: {Path(file_path).name}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )

    print(f"Records loaded : {df.count():,}")
    print(f"Columns        : {df.columns}")
    return df

bronze_df = load_bronze_transactions(BRONZE_PATH)
bronze_df.printSchema()

LOADING BRONZE TRANSACTION DATA
Source: C:\FinPulse Project\data\bronze\transactions
Layout: partitioned  |  loading latest partition: date=2026-03-29
Records loaded : 6,362,620
Columns        : ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'ingestion_timestamp', 'data_source', 'pipeline_version']
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- data_source: string (nullable = true)
 |-- pipeline_version: string (nu

In [6]:
def apply_silver_transformations(df):
    """Applies all Silver layer transformations."""
    print("=" * 60)
    print("APPLYING SILVER TRANSFORMATIONS")
    print("=" * 60)

    # Step 1: Remove zero amount transactions
    initial_count = df.count()
    df = df.filter(F.col("amount") > 0)
    removed = initial_count - df.count()
    print(f"Step 1 - Removed {removed:,} zero amount transactions")

    # Step 2: Add balance discrepancy flag
    df = df.withColumn(
        "is_balance_discrepancy",
        F.when(
            (F.col("type").isin(["TRANSFER", "CASH_OUT"]))
            & (F.col("oldbalanceOrg") > 0)
            & (
                F.round(F.col("oldbalanceOrg") - F.col("amount"), 2)
                != F.round(F.col("newbalanceOrig"), 2)
            ),
            True,
        ).otherwise(False),
    )
    discrepancies = df.filter(F.col("is_balance_discrepancy")).count()
    print(f"Step 2 - Balance discrepancies flagged: {discrepancies:,}")

    # Step 3: Convert step to simulation time fields
    df = df.withColumn("hour_of_simulation", F.col("step").cast(IntegerType()))
    df = df.withColumn("day_of_simulation", (F.col("step") / 24).cast(IntegerType()) + 1)
    print("Step 3 - Added hour_of_simulation and day_of_simulation")

    # Step 4: Encode transaction type
    df = df.withColumn(
        "type_encoded",
        F.when(F.col("type") == "TRANSFER", 0)
        .when(F.col("type") == "CASH_OUT", 1)
        .when(F.col("type") == "PAYMENT", 2)
        .when(F.col("type") == "DEBIT", 3)
        .when(F.col("type") == "CASH_IN", 4)
        .otherwise(-1),
    )
    print("Step 4 - Transaction type encoded")

    # Step 5: Add risk flag
    mean_amount = df.agg(F.mean("amount")).collect()[0][0]
    df = df.withColumn(
        "risk_flag",
        F.when(
            (F.col("type").isin(["TRANSFER", "CASH_OUT"]))
            & (F.col("amount") > mean_amount),
            True,
        ).otherwise(False),
    )
    risk_flagged = df.filter(F.col("risk_flag")).count()
    print(f"Step 5 - Risk flagged transactions: {risk_flagged:,}")

    # Step 6: Add balance difference column
    df = df.withColumn(
        "balance_diff",
        F.col("oldbalanceOrg") - F.col("newbalanceOrig"),
    )
    print("Step 6 - Added balance_diff column")

    # Step 7: Deduplicate
    before_dedup = df.count()
    df = df.dropDuplicates(["nameOrig", "step", "amount", "type"])
    after_dedup = df.count()
    print(f"Step 7 - Deduplicated transactions: {before_dedup - after_dedup:,} duplicates")

    # Step 8: Add pipeline metadata
    df = df.withColumn("silver_processed_at", F.lit(INGESTION_TIMESTAMP))
    df = df.withColumn("pipeline_version", F.lit(PIPELINE_VERSION))
    print("Step 8 - Added pipeline metadata")

    print("\nSilver transformations complete")
    print(f"Final record count: {df.count():,}")
    print(f"Final columns: {len(df.columns)}")

    return df


silver_df = apply_silver_transformations(bronze_df)
silver_df.printSchema()

APPLYING SILVER TRANSFORMATIONS
Step 1 - Removed 16 zero amount transactions
Step 2 - Balance discrepancies flagged: 1,199,155
Step 3 - Added hour_of_simulation and day_of_simulation
Step 4 - Transaction type encoded
Step 5 - Risk flagged transactions: 1,326,648
Step 6 - Added balance_diff column
Step 7 - Deduplicated transactions: 0 duplicates
Step 8 - Added pipeline metadata

Silver transformations complete
Final record count: 6,362,604
Final columns: 21
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- data_source: s

In [7]:
def validate_silver_data(df):
    """
    Basic validation checks before writing to Silver layer.
    Catches issues before bad data reaches Gold layer.
    """
    print("=" *60)
    print("SILVER DATA VALIDATION")
    print("="*60)

    checks_passed = 0
    checks_failed = 0
    # Check 1: No zero amounts
    zero_amounts = df.filter(F.col("amount")<=0).count()
    if zero_amounts == 0:
        print(" Check 1 PASSED - No zero amount transactions")
        checks_passed += 1
    else:
        print(f" Check 1 FAILED - {zero_amounts} zero amounts found")
        checks_failed += 1
    # Check 2: isFraud only 0 or 1
    invalid_fraud = df.filter( ~F.col("isFraud").isin([0,1])).count()
    if invalid_fraud == 0:
        print(" Check 2 PASSED - isFraud values valid")
        checks_passed += 1
    else:
        print(f" Check 2 FAILED — {invalid_fraud} invalid fraud values")
        checks_failed += 1
    #Check 3 - No null amounts
    null_amounts = df.filter(F.col("amount").isNull()).count()
    if null_amounts == 0:
        print(" Check 3 PASSED - No null amounts")
        checks_passed += 1
    else:
        print(f" Check 3 FAILED - {null_amounts} null amounts found")
        checks_failed += 1
    
    # Check 4 - Balance discrepancy fraud correlation
    discrepancy_fraud_rate = df.filter(F.col("is_balance_discrepancy")==True).agg(F.mean("isFraud")).collect()[0][0]
    print(f" Check 4 INFO — Fraud rate in discrepancy records: {discrepancy_fraud_rate:.2%}")
    checks_passed += 1
    print(f"\nValidation complete — "
          f"{checks_passed} passed, {checks_failed} failed")
    
    if checks_failed > 0:
        raise Exception(f" {checks_failed} validation checks failed. "
                        f"Pipeline stopped.")
     
    return True

validate_silver_data(silver_df)








SILVER DATA VALIDATION
 Check 1 PASSED - No zero amount transactions
 Check 2 PASSED - isFraud values valid
 Check 3 PASSED - No null amounts
 Check 4 INFO — Fraud rate in discrepancy records: 0.00%

Validation complete — 4 passed, 0 failed


True

In [8]:
def write_silver_iceberg(df, iceberg_warehouse):
    """
    Writes Silver data as Apache Iceberg table.
    Gives ACID transactions, time travel, schema evolution.
    Stored on Google Drive via Hadoop catalog.
    """
    print("=" * 60)
    print("WRITING SILVER LAYER — Apache Iceberg")
    print("=" * 60)

    # Ensure catalog warehouse points to correct location
    spark.conf.set("spark.sql.catalog.local.warehouse", iceberg_warehouse)

    # Create namespace if not exists
    spark.sql("CREATE NAMESPACE IF NOT EXISTS local.finpulse")
    print("✅ Namespace local.finpulse ready")

    # Write as Iceberg table
    df.writeTo("local.finpulse.silver_transactions") \
      .tableProperty("format-version", "2") \
      .tableProperty("write.parquet.compression-codec", "snappy") \
      .tableProperty("history.expire.min-snapshots-to-keep", "3") \
      .createOrReplace()

    print("✅ Iceberg table written: local.finpulse.silver_transactions")

    # ── Verify Iceberg features ───────────────────────────────

    # 1. Row count verification
    count = spark.table("local.finpulse.silver_transactions").count()
    print(f"✅ Records in Iceberg table: {count:,}")

    # 2. Show Iceberg snapshots — proves time travel is working
    print("\n── Iceberg Snapshots (Time Travel) ──")
    spark.sql("""
        SELECT snapshot_id, committed_at, operation, summary
        FROM local.finpulse.silver_transactions.snapshots
    """).show(truncate=False)

    # 3. Show Iceberg history
    print("── Iceberg History ──")
    spark.sql("""
        SELECT *
        FROM local.finpulse.silver_transactions.history
    """).show(truncate=False)

    # 4. Show schema
    print("── Iceberg Table Schema ──")
    spark.sql("""
        DESCRIBE TABLE local.finpulse.silver_transactions
    """).show(truncate=False)

    return "local.finpulse.silver_transactions"

output = write_silver_iceberg(silver_df, ICEBERG_WAREHOUSE)


WRITING SILVER LAYER — Apache Iceberg
✅ Namespace local.finpulse ready
✅ Iceberg table written: local.finpulse.silver_transactions
✅ Records in Iceberg table: 6,362,604

── Iceberg Snapshots (Time Travel) ──
+-------------------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_id        |committed_at           |operation|summary                                                                                                                                                                                                                                                                                                                 |
+-------------------+-----------------------+---------

In [9]:
# ============================================================
# APACHE ICEBERG FEATURE DEMONSTRATION
# ============================================================
print("=" * 60)
print("APACHE ICEBERG FEATURE DEMONSTRATION")
print("=" * 60)

# Feature 1 — Time Travel
print("\n📅 Feature 1 — Time Travel")
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at
    FROM local.finpulse.silver_transactions.snapshots
    ORDER BY committed_at
""").collect()

if snapshots:
    snapshot_id = snapshots[0]["snapshot_id"]
    time_travel_df = spark.read \
        .format("iceberg") \
        .option("snapshot-id", snapshot_id) \
        .load("local.finpulse.silver_transactions")
    print(f"Records at snapshot {snapshot_id}: {time_travel_df.count():,}")
    print("✅ Time travel working")

# Feature 2 — Schema Evolution
print("\n🔄 Feature 2 — Schema Evolution")
try:
    spark.sql("""
        ALTER TABLE local.finpulse.silver_transactions
        ADD COLUMN risk_score DOUBLE
    """)
    print("✅ Added risk_score column — zero data rewrite needed")
except Exception:
    print("ℹ️  risk_score column already exists — skipping (safe to re-run)")

# Feature 3 — ACID Proof
print("\n🔒 Feature 3 — ACID Transaction Proof")
spark.sql("""
    SELECT COUNT(*) as total,
           SUM(CAST(isFraud AS INT)) as fraud_count,
           ROUND(AVG(amount), 2) as avg_amount
    FROM local.finpulse.silver_transactions
""").show()
print("✅ ACID guaranteed")

# Feature 4 — Snapshots
print("\n📸 Feature 4 — Snapshot History")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM local.finpulse.silver_transactions.snapshots
""").show(truncate=False)

print("\n" + "=" * 60)
print("✅ ICEBERG FEATURES DEMONSTRATED")
print("   Time Travel     → Financial audit compliance")
print("   Schema Evolution → Add columns without downtime")
print("   ACID            → No partial writes ever")
print("   Snapshots       → Complete write history")
print("=" * 60)

APACHE ICEBERG FEATURE DEMONSTRATION

📅 Feature 1 — Time Travel
Records at snapshot 8040436300429848040: 6,362,604
✅ Time travel working

🔄 Feature 2 — Schema Evolution
✅ Added risk_score column — zero data rewrite needed

🔒 Feature 3 — ACID Transaction Proof
+-------+-----------+----------+
|  total|fraud_count|avg_amount|
+-------+-----------+----------+
|6362604|       8197| 179862.36|
+-------+-----------+----------+

✅ ACID guaranteed

📸 Feature 4 — Snapshot History
+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|8040436300429848040|2026-04-06 20:18:59.28 |overwrite|
|4867979137453193770|2026-04-07 14:32:06.208|overwrite|
|8014140024838959978|2026-04-07 15:38:37.282|overwrite|
+-------------------+-----------------------+---------+


✅ ICEBERG FEATURES DEMONSTRATED
   Time Travel     → Financial audit compliance
   Schema Evolution → Add columns without downtime

In [10]:
print("=" * 60)
print("FINPULSE SILVER LAYER — COMPLETE SUMMARY")
print("=" * 60)
print(f"Source: Bronze transactions")
print(f"Records processed: {silver_df.count():,}")
print(f"Columns: {len(silver_df.columns)}")
print(f"New columns added:")
print(f"  - is_balance_discrepancy")
print(f"  - hour_of_simulation")
print(f"  - day_of_simulation")
print(f"  - type_encoded")
print(f"  - risk_flag")
print(f"  - balance_diff")
print(f"  - silver_processed_at")
print(f"  - pipeline_version")
print(f"Output: {output}")
print("=" * 60)
print("✅ SILVER LAYER COMPLETE — Ready for Soda Core checks")
print("=" * 60)

spark.stop()

FINPULSE SILVER LAYER — COMPLETE SUMMARY
Source: Bronze transactions
Records processed: 6,362,604
Columns: 21
New columns added:
  - is_balance_discrepancy
  - hour_of_simulation
  - day_of_simulation
  - type_encoded
  - risk_flag
  - balance_diff
  - silver_processed_at
  - pipeline_version
Output: local.finpulse.silver_transactions
✅ SILVER LAYER COMPLETE — Ready for Soda Core checks


## Cloud Persistence — Google Drive Sync


In [11]:
# ============================================================
# SYNC ICEBERG TABLE TO GOOGLE DRIVE
# Runs AFTER Spark write completes — avoids sync conflicts
# ============================================================
import shutil
from pathlib import Path

LOCAL_ICEBERG = Path(r"C:\FinPulse Project\data\silver\iceberg_warehouse")
DRIVE_ICEBERG = Path(r"G:\My Drive\FinPulse\data\silver\iceberg_warehouse")

print("=" * 60)
print("SYNCING ICEBERG TABLE TO GOOGLE DRIVE")
print("=" * 60)

if LOCAL_ICEBERG.exists():
    # Remove old Drive copy if exists
    if DRIVE_ICEBERG.exists():
        shutil.rmtree(DRIVE_ICEBERG)
        print("✅ Removed old Drive copy")

    # Copy fresh Iceberg table to Drive
    shutil.copytree(LOCAL_ICEBERG, DRIVE_ICEBERG)
    print(f"✅ Iceberg table synced to: {DRIVE_ICEBERG}")

    # Calculate sizes
    local_size = sum(f.stat().st_size for f in LOCAL_ICEBERG.rglob('*') if f.is_file())
    print(f"   Table size: {local_size / (1024*1024):.1f} MB")
    print("\n✅ Google Drive backup complete")
    print("   Local copy retained for next pipeline run")
else:
    print("❌ Local Iceberg table not found")


SYNCING ICEBERG TABLE TO GOOGLE DRIVE
✅ Removed old Drive copy
✅ Iceberg table synced to: G:\My Drive\FinPulse\data\silver\iceberg_warehouse
   Table size: 925.9 MB

✅ Google Drive backup complete
   Local copy retained for next pipeline run
